# Selecting atoms with categories

Many sisl methods take a set of atoms: `Geometry.sub`, `Geometry.remove`, `Geometry.translate`,
plotting styles and more. Atomic indices work, but they change as soon as the geometry changes.
Atom *categories* select atoms by what they are, where they are, and what surrounds them.
The same category works on a small test system and on a large production geometry.

This tutorial introduces categories one step at a time:

1. one category and its result,
2. categories where sisl expects atoms,
3. combining categories,
4. the atomic environment (neighbors),
5. the position of atoms,
6. the short keyword form,
7. changing a geometry with categories.

In [ ]:
import numpy as np
import sisl as si
from sisl.geom import AtomCategory, AtomNeighbors, AtomXYZ, AtomZ

Our test system is a zigzag graphene nanoribbon. It is periodic along the ribbon and has two edges.
We replace two carbon atoms with nitrogen, one at an edge and one inside the ribbon,
and we remove one atom to make a vacancy.

In [ ]:
ribbon = si.geom.zgnr(6).tile(4, 0)
ribbon.atoms[[17, 20]] = si.Atom("N", R=1.4342)
ribbon = ribbon.remove(38)
print(ribbon)

In [ ]:
ribbon.plot(axes="xy")

## 1. One category

`AtomZ` selects atoms by their atomic number. It accepts the number or the symbol.
The method `mask` returns one boolean value per atom: `True` for the atoms in the category.

In [ ]:
nitrogen = AtomZ("N")
mask = nitrogen.mask(ribbon)
print(mask)
print("indices:", np.flatnonzero(mask))

A category does not store the geometry. You can use the same `nitrogen` category on any geometry.

## 2. Use a category where sisl expects atoms

Methods that take atomic indices also take a category.
Here we extract the nitrogen atoms, and we highlight them in a plot.

In [ ]:
print(ribbon.sub(nitrogen))

In [ ]:
ribbon.plot(axes="xy", atoms_style=[{"atoms": nitrogen, "color": "blue"}])

## 3. Combine categories

Categories combine with the operators of Python sets:

| Operator | Meaning |
|---|---|
| `A & B` | in `A` and in `B` |
| `A \| B` | in `A` or in `B` |
| `A ^ B` | in exactly one of `A` and `B` |
| `~A` | not in `A` |

For example, all atoms that are not carbon, or the carbon atoms only:

In [ ]:
carbon = AtomZ("C")
print("not carbon:", np.flatnonzero((~carbon).mask(ribbon)))
print("carbon count:", carbon.mask(ribbon).sum(), "of", ribbon.na)

## 4. The atomic environment

`AtomNeighbors` counts the neighbors of each atom within a radius `R` (in Ångström).
Atoms in the bulk of graphene have 3 neighbors.
The atoms at the edges, and the atoms around the vacancy, have only 2.

In [ ]:
two_bonds = AtomNeighbors(max=2, R=1.5)
print(np.flatnonzero(two_bonds.mask(ribbon)))

In [ ]:
ribbon.plot(axes="xy", atoms_style=[{"atoms": two_bonds, "color": "red"}])

`AtomNeighbors` takes one or two numbers: `AtomNeighbors(3)` means exactly 3 neighbors,
`AtomNeighbors(1, 2)` means 1 or 2 neighbors. The keywords `min` and `max` give only one bound.

The argument `neighbor` counts only the neighbors in another category.
These are the atoms with at least one nitrogen neighbor:

In [ ]:
next_to_N = AtomNeighbors(min=1, neighbor=nitrogen, R=1.5)
print(np.flatnonzero(next_to_N.mask(ribbon)))

## 5. The position of atoms

`AtomXYZ` selects atoms by their coordinates. A tuple `(min, max)` gives an interval,
and `None` means no bound. The atoms near the vacancy are not at the edges,
so a position interval separates them from the edge atoms:

In [ ]:
y = ribbon.xyz[:, 1]
inside = AtomXYZ(y=(y.min() + 1, y.max() - 1))
vacancy = two_bonds & inside
print(np.flatnonzero(vacancy.mask(ribbon)))

`AtomXYZ` also has a short form with `<` and `>`, also for fractional coordinates (`fx`, `fy`, `fz`):

In [ ]:
lower_half = AtomXYZ.fy < 0.5
print(lower_half.mask(ribbon).sum(), "atoms in the lower half of the cell")

In [ ]:
ribbon.plot(
    axes="xy",
    atoms_style=[
        {"atoms": two_bonds & ~inside, "color": "red"},
        {"atoms": vacancy, "color": "green"},
    ],
)

## 6. The keyword form

`AtomCategory` builds categories from keywords. Each keyword is the name of a category
without `Atom`, and many keywords combine with `&`.
A `dict` with the same keywords works directly as an index.

In [ ]:
edge_carbon = AtomCategory(Z="C", neighbors={"max": 2, "R": 1.5})
print(edge_carbon)
print(np.flatnonzero(edge_carbon.mask(ribbon)))
print(ribbon.sub({"Z": "N"}).na, "nitrogen atoms")

An unknown keyword shows the valid keywords:

In [ ]:
try:
    AtomCategory(neigbors=2)
except ValueError as e:
    print(e)

## 7. Change a geometry with categories

A category describes *which* atoms to change. The method describes *how*.
Here we move the atoms next to nitrogen out of the plane:

In [ ]:
buckled = ribbon.translate([0, 0, 0.3], atoms=next_to_N)
print(np.unique(buckled.xyz[:, 2] - ribbon.xyz[:, 2]))

`sisl.geom.saturate` adds atoms to the missing bonds.
Each rule is a pair of a category and a `Saturate` object, and the first rule that
selects an atom wins. Here the edge carbon atoms get hydrogen, and the edge nitrogen atom
gets nothing because it already has a lone pair:

In [ ]:
from sisl.geom import Saturate, saturate

edge = two_bonds & ~inside
passivated = saturate(
    ribbon,
    [
        (edge & nitrogen, Saturate("H", coordination=2, R=1.5)),
        (edge, Saturate("H", bond=1.09, R=1.5)),
    ],
)
print(passivated.na - ribbon.na, "hydrogen atoms added")

In [ ]:
passivated.plot(axes="xy")

`Saturate` finds the number of missing bonds from the most frequent number of bonds of the
same element, here 3 for carbon. The argument `coordination` sets it directly:
`coordination=2` for the nitrogen means that it is already saturated.
Without `bond`, the bond length is the sum of the atomic radii, which is approximate.

## Summary

- A category selects atoms by element, position or environment, and it does not depend on the indices.
- `mask` returns one boolean per atom. Methods that take atoms also take a category or a `dict`.
- Combine categories with `&`, `|`, `^` and `~`.
- `AtomCategory(...)` and plain `dict`s build categories from keywords.

The reference of all categories is in the API documentation: [Indexing atoms](../api/geom/indexing.rst).